# 1.           Scraping and Merging NFL Data (For 2025)

In [5]:
# importing necessary Libraries
import numpy as np
import pandas as pd
import random
import time

In [6]:
# creating list of teams
teams = ['crd', 'atl', 'rav', 'buf', 'car', 'chi', 'cin', 'cle', 'dal', 'den', 'det', 'gnb', 'htx', 'clt', 'jax', 'kan', 'sdg',
        'ram', 'rai', 'mia', 'min', 'nwe', 'nor', 'nyg', 'nyj', 'phi', 'pit', 'sea', 'sfo', 'tam', 'oti', 'was']
print(f'number of teams={len(teams)}')

number of teams=32


In [7]:
# creating Dictionary to rename some of the columns

rename_dict = { 'Unamed: 5': 'Home', 'Rslt': 'Win', 'Pts': 'Tm_Pts', 'PtsO': 'Opp_Pts',
                 'Cmp': 'pCmp', 'Att': 'pAtt', 'Cmp%': 'pCmp%', 'Yds': 'pYds',
                 'TD': 'pTD', 'Y/A': 'pY/A', 'AY/A': 'pAY/A', 'Rate': 'pRate',
                 'Yds.1': 'SkYds', 'Att.1': 'rAtt', 'Yds.2': 'rYds', 'TD.1': 'rTD',
                 'Y/A.1': 'rY/A', 'Yds.3': 'PntYds', 'Pass': 'fdPass', 'Rsh': 'fdRush',
                 'Pen': 'fdPen', 'Pen.1': 'Pen', 'Yds.4': 'PenYds'}

In [13]:
start_time = time.time()

# list of seasons to download
seasons = range(2015, 2025)

# empty dataframe to append
nfl_df = pd.DataFrame()

# iterate through the seasons
for season in seasons:
    # iterate through the teams
    for team in teams:
        # set the URL
        url = 'https://www.pro-football-reference.com/teams/' + team + '/' + str(season) + '/gamelog/'
        print(url)

        # get team gamelog data
        table_id = 'table_pfr_team-year_game-logs_team-year-regular-season-game-log'
        tm_df = pd.read_html(url, header=1, attrs={'id':table_id})[0]
        tm_df = tm_df.dropna(subset=['Rk']).rename(rename_dict, axis=1)

        # get opponent gamelog data
        table_id = 'table_pfr_team-year_game-logs_team-year-regular-season-opponent-game-log'
        opp_df = pd.read_html(url, header=1, attrs={'id':table_id})[0]
        opp_df = opp_df.dropna(subset=['Rk']).rename(rename_dict, axis=1)
       
        merge_on_cols = tm_df.columns[:11].tolist()
        
        # merge the dataframes. The statistical columns (from index 11 onwards) 
        # will automatically get the suffixes '_Tm' and '_Opp'
        merged_df = pd.merge(
            tm_df, 
            opp_df, 
            on=merge_on_cols, 
            suffixes=('_Tm', '_Opp')

        # find all columns that ended up with the new suffixes
        tm_cols = [col for col in merged_df.columns if col.endswith('_Tm')]
        opp_cols = [col for col in merged_df.columns if col.endswith('_Opp')]
        
        # create a final renaming dictionary to remove the suffixes and add your desired 'Tm_' or 'Opp_' prefix
        final_rename_dict = {}
        for col in tm_cols:
            base_col = col.replace('_Tm', '')
            final_rename_dict[col] = f'Tm_{base_col}'
            
        for col in opp_cols:
            base_col = col.replace('_Opp', '')
            final_rename_dict[col] = f'Opp_{base_col}'

        merged_df = merged_df.rename(columns=final_rename_dict)
        
        # insert Season and Team as new columns
        merged_df.insert(loc=0, column='Season', value=season)
        merged_df.insert(loc=1, column='Team', value=team.upper())

        # concatenate the team gamelog to the aggregate dataframe
        nfl_df = pd.concat([nfl_df, merged_df], ignore_index=True)

        # IMPORTANT: Learning from the first model, I had to put a timer to not go to internet jail
        time.sleep(random.randint(4, 5))
end_time = time.time()
print(f'Elapsed time: {end_time - start_time:1f} seconds')
print(nfl_df.info())

https://www.pro-football-reference.com/teams/crd/2015/gamelog/
https://www.pro-football-reference.com/teams/atl/2015/gamelog/
https://www.pro-football-reference.com/teams/rav/2015/gamelog/
https://www.pro-football-reference.com/teams/buf/2015/gamelog/
https://www.pro-football-reference.com/teams/car/2015/gamelog/
https://www.pro-football-reference.com/teams/chi/2015/gamelog/
https://www.pro-football-reference.com/teams/cin/2015/gamelog/
https://www.pro-football-reference.com/teams/cle/2015/gamelog/
https://www.pro-football-reference.com/teams/dal/2015/gamelog/
https://www.pro-football-reference.com/teams/den/2015/gamelog/
https://www.pro-football-reference.com/teams/det/2015/gamelog/
https://www.pro-football-reference.com/teams/gnb/2015/gamelog/
https://www.pro-football-reference.com/teams/htx/2015/gamelog/
https://www.pro-football-reference.com/teams/clt/2015/gamelog/
https://www.pro-football-reference.com/teams/jax/2015/gamelog/
https://www.pro-football-reference.com/teams/kan/2015/g

In [18]:
# clean the data

# 1. Rename the 'Unnamed: 5' column to 'Home'
nfl_df = nfl_df.rename(columns={'Unnamed: 5': 'Home_Marker'})

# 2. Convert 'Home_Marker' to the binary 'Home' column (1 for Home, 0 for Away)
nfl_df['Home'] = np.where(nfl_df['Home_Marker'] == '@', 0, 1)

# 3. Convert 'Win' and 'OT' columns to binary (these names exist)
# Win/Loss: 'W' is a Win (1), otherwise (Loss/Tie) is 0.
nfl_df['Win'] = np.where(nfl_df['Win'] == 'W', 1, 0)

# Overtime: 'OT' is Overtime (1), otherwise is 0.
nfl_df['OT'] = np.where(nfl_df['OT'] == 'OT', 1, 0)

# 4. Drop the now-redundant marker column
nfl_df = nfl_df.drop(columns=['Home_Marker'], axis=1)

print(nfl_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5246 entries, 0 to 5245
Data columns (total 86 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Season      5246 non-null   int64  
 1   Team        5246 non-null   object 
 2   Gtm         5246 non-null   float64
 3   Week        5246 non-null   float64
 4   Date        5246 non-null   object 
 5   Day         5246 non-null   object 
 6   Opp         5246 non-null   object 
 7   Win         5246 non-null   int64  
 8   Tm_Pts      5246 non-null   int64  
 9   Opp_Pts     5246 non-null   int64  
 10  OT          5246 non-null   int64  
 11  Tm_pCmp     5246 non-null   int64  
 12  Tm_pAtt     5246 non-null   int64  
 13  Tm_pCmp%    5246 non-null   float64
 14  Tm_pYds     5246 non-null   int64  
 15  Tm_pTD      5246 non-null   int64  
 16  Tm_pY/A     5246 non-null   float64
 17  Tm_pAY/A    5246 non-null   float64
 18  Tm_pRate    5246 non-null   float64
 19  Tm_Sk       5246 non-null  

In [20]:
# SAVE THE DATA TO CSV
nfl_df.to_csv('nfl_gamelogs_2015-2024_NEW.csv', index=False)

# 2. Scrape NFL Vegas Lines Data

In [52]:
# Scrape the NFL vegas lines data
# this code is more or less the exact same for the team stats, however this is for the vegas lines

start_time = time.time()

seasons = range(2015, 2025)

veg_df = pd.DataFrame()


# Iterate through the seasons
for season in seasons:
    # Iterate through the teams
    for team in teams:
        # Set the URL
        url = 'https://www.pro-football-reference.com/teams/' + team + '/' + str(season) + '_lines.htm'
        print(url)

        # get vegas lines from table
        lines_df = pd.read_html(url, header=0, attrs={'id':'vegas_lines'})[0]

        # insert the season and team columns
        lines_df.insert(loc=0, column='Season', value=season)
        lines_df.insert(loc=1, column='Team', value=team.upper())

        # concat the team lines df to the aggregate df
        veg_df = pd.concat([veg_df, lines_df], ignore_index=True)

        # Again, internet jail is bad so have to put a timer
        time.sleep(random.randint(4, 5))

end_time = time.time()

elapsed_time = end_time - start_time
print(f'Elapsed time: {elapsed_time} seconds')
print(veg_df.info())


https://www.pro-football-reference.com/teams/crd/2015_lines.htm
https://www.pro-football-reference.com/teams/atl/2015_lines.htm
https://www.pro-football-reference.com/teams/rav/2015_lines.htm
https://www.pro-football-reference.com/teams/buf/2015_lines.htm
https://www.pro-football-reference.com/teams/car/2015_lines.htm
https://www.pro-football-reference.com/teams/chi/2015_lines.htm
https://www.pro-football-reference.com/teams/cin/2015_lines.htm
https://www.pro-football-reference.com/teams/cle/2015_lines.htm
https://www.pro-football-reference.com/teams/dal/2015_lines.htm
https://www.pro-football-reference.com/teams/den/2015_lines.htm
https://www.pro-football-reference.com/teams/det/2015_lines.htm
https://www.pro-football-reference.com/teams/gnb/2015_lines.htm
https://www.pro-football-reference.com/teams/htx/2015_lines.htm
https://www.pro-football-reference.com/teams/clt/2015_lines.htm
https://www.pro-football-reference.com/teams/jax/2015_lines.htm
https://www.pro-football-reference.com/t

In [55]:
# Clean the data

veg_df = veg_df.drop(veg_df.columns[6:], axis=1)

veg_df = veg_df.rename(columns={'G#': 'Gtm', 'Over/Under':'Total'})

veg_df = veg_df.query ('(Season <= 2020 and Gtm < 17) or (Season >= 2021 and Gtm < 18)')

print(veg_df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 5246 entries, 0 to 5482
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Season  5246 non-null   int64  
 1   Team    5246 non-null   object 
 2   Gtm     5246 non-null   int64  
 3   Opp     5246 non-null   object 
 4   Spread  5246 non-null   float64
 5   Total   5246 non-null   float64
dtypes: float64(2), int64(2), object(2)
memory usage: 286.9+ KB
None


In [69]:
# SAVE DATA TO CSV
veg_df.to_csv('nfl_vegas_lines_2015-2024_NEW.csv', index=False)

# 3. Merge Gamelog Data with Vegas Lines Data

In [70]:
nfl_df = pd.read_csv('nfl_gamelogs_2015-2024_NEW.csv')
veg_df = pd.read_csv('nfl_vegas_lines_2015-2024_NEW.csv')


In [71]:
#checking that the two dfs contain the same number of rows
print(nfl_df.shape)
print(veg_df.shape)

(5246, 86)
(5246, 6)


In [72]:
# merge the datasets based on the following three columns forming a unique "key"
merged_df = pd.merge(nfl_df, veg_df, on=['Season', 'Team', 'Gtm'])
print(merged_df.shape)

(5246, 89)


In [73]:
# create cover and over columns
merged_df['Cover'] =np.where(merged_df['Tm_Pts'] + merged_df['Spread'] > merged_df['Opp_Pts'], 1, 0)
merged_df['Over'] =np.where(merged_df['Tm_Pts'] + merged_df['Opp_Pts'] > merged_df['Total'], 1, 0)

print(merged_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5246 entries, 0 to 5245
Data columns (total 91 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Season      5246 non-null   int64  
 1   Team        5246 non-null   object 
 2   Gtm         5246 non-null   float64
 3   Week        5246 non-null   float64
 4   Date        5246 non-null   object 
 5   Day         5246 non-null   object 
 6   Opp_x       5246 non-null   object 
 7   Win         5246 non-null   int64  
 8   Tm_Pts      5246 non-null   int64  
 9   Opp_Pts     5246 non-null   int64  
 10  OT          5246 non-null   int64  
 11  Tm_pCmp     5246 non-null   int64  
 12  Tm_pAtt     5246 non-null   int64  
 13  Tm_pCmp%    5246 non-null   float64
 14  Tm_pYds     5246 non-null   int64  
 15  Tm_pTD      5246 non-null   int64  
 16  Tm_pY/A     5246 non-null   float64
 17  Tm_pAY/A    5246 non-null   float64
 18  Tm_pRate    5246 non-null   float64
 19  Tm_Sk       5246 non-null  

In [62]:
# Print an example to verify merge worked
print(merged_df.query('Season == 2022 and Team == "ATL"'))

      Season Team   Gtm  Week        Date  Day Opp_x  Win  Tm_Pts  Opp_Pts  \
3633    2022  ATL   1.0   1.0  2022-09-11  Sun   NOR    0      26       27   
3634    2022  ATL   2.0   2.0  2022-09-18  Sun   LAR    0      27       31   
3635    2022  ATL   3.0   3.0  2022-09-25  Sun   SEA    1      27       23   
3636    2022  ATL   4.0   4.0  2022-10-02  Sun   CLE    1      23       20   
3637    2022  ATL   5.0   5.0  2022-10-09  Sun   TAM    0      15       21   
3638    2022  ATL   6.0   6.0  2022-10-16  Sun   SFO    1      28       14   
3639    2022  ATL   7.0   7.0  2022-10-23  Sun   CIN    0      17       35   
3640    2022  ATL   8.0   8.0  2022-10-30  Sun   CAR    1      37       34   
3641    2022  ATL   9.0   9.0  2022-11-06  Sun   LAC    0      17       20   
3642    2022  ATL  10.0  10.0  2022-11-10  Thu   CAR    0      15       25   
3643    2022  ATL  11.0  11.0  2022-11-20  Sun   CHI    1      27       24   
3644    2022  ATL  12.0  12.0  2022-11-27  Sun   WAS    0      1

In [74]:
nfl_df = nfl_df.rename(columns={'Opp_x':'Opp'})

In [75]:
merged_df.to_csv('nfl_gamelogs_vegas_2015-2024_NEW.csv', index=False)

# 4. Win Percentage for Home Teams Favored from -7.0 to -6.5

small note before code: last year This particualar trend had a hit rate of 75%(!!)

In [76]:
# reload the save df as nfl_df
nfl_df = pd.read_csv('nfl_gamelogs_vegas_2015-2024_NEW.csv')

In [77]:
# determine win percentage for home teams with the above trend

# first get the home favorites
home_fav_df = nfl_df.query('Home == 1 and -7.0 <= Spread <= -6.5')
home_fav_count = len(home_fav_df)

#next get the wins from the home favorites
home_win_df = home_fav_df.query('Win == 1')
home_win_count = len(home_win_df)

# display the precentage
print(f'Win percentage for home teams favored from -7.0 to -6.5: {home_win_count / home_fav_count: .2%} ({home_win_count} of {home_fav_count})')

Win percentage for home teams favored from -7.0 to -6.5:  76.92% (140 of 182)
